In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!nvidia-smi

In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/abdelrahmantarekm"))

In [ ]:
import os

base_path = "/kaggle/input/datasets/abdelrahmantarekm/final-merged-dataset/final_dataset"

train_path = f"{base_path}/train"

print("Train folder contents:", os.listdir(train_path))

In [ ]:
import os
import shutil
import random

# Base path (Kaggle)
base_path = "/kaggle/input/datasets/abdelrahmantarekm/final-merged-dataset/final_dataset"

# Source folders
paths_real = [
    f"{base_path}/train/real",
    f"{base_path}/validation/real",
    f"{base_path}/test/real"
]

paths_fake = [
    f"{base_path}/train/fake",
    f"{base_path}/validation/fake",
    f"{base_path}/test/fake"
]

# Destination subset folder (IMPORTANT FIX)
subset_path = "/kaggle/working/subset_dataset_10k"
os.makedirs(os.path.join(subset_path, "Real"), exist_ok=True)
os.makedirs(os.path.join(subset_path, "Fake"), exist_ok=True)

def copy_random_combined(src_folders, dst_folder, n):
    all_files = []
    for folder in src_folders:
        files = [os.path.join(folder, f) for f in os.listdir(folder)]
        all_files.extend(files)
    random.shuffle(all_files)
    for f in all_files[:n]:
        shutil.copy(f, dst_folder)

# Copy 5000 images each for balanced 10K subset
copy_random_combined(paths_real, os.path.join(subset_path, "Real"), 5000)
copy_random_combined(paths_fake, os.path.join(subset_path, "Fake"), 5000)

# Verify counts
print("Subset Real images:", len(os.listdir(os.path.join(subset_path, "Real"))))
print("Subset Fake images:", len(os.listdir(os.path.join(subset_path, "Fake"))))

In [ ]:
!pip install grad-cam

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from pytorch_grad_cam import GradCAM
import matplotlib.pyplot as plt
import numpy as np
import os
import random

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
subset_path = "/kaggle/working/subset_dataset_10k"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# Full dataset
full_dataset = datasets.ImageFolder(subset_path, transform=transform)

# Split 70/15/15 for train/val/test
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_set, val_set, test_set = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 16
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [ ]:
def train_fsfd_full(model, train_loader, val_loader, test_loader,
                    epochs=20, patience=5):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):

        # =====================
        # TRAIN
        # =====================
        model.train()
        running_loss, correct, total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # =====================
        # VALIDATION
        # =====================
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)

                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(1)

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f}")

        # =====================
        # EARLY STOPPING
        # =====================
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_fsfdnet.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered")
                break

    # =====================
    # LOAD BEST MODEL
    # =====================
    model.load_state_dict(torch.load("best_fsfdnet.pth"))
    model.eval()

    # =====================
    # TEST PHASE (FIXED RETURN)
    # =====================
    all_preds, all_labels, all_probs = [], [], []
    all_features, sample_images = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)

            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            sample_images.extend(imgs.cpu().numpy())

            # ===== FEATURES =====
            spatial_feat = model.spatial_model(imgs)

            x_gray = torch.mean(imgs, dim=1, keepdim=True)
            fft = torch.fft.fftshift(torch.fft.fft2(x_gray))
            fft_mag = torch.log(torch.abs(fft) + 1e-8)

            freq_feat = model.freq_conv(fft_mag).view(imgs.size(0), -1)

            fused = torch.cat([spatial_feat, freq_feat], dim=1)

            all_features.extend(fused.cpu().numpy())

    # =====================
    # METRICS
    # =====================
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Real','Fake']))

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)

    test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Test Accuracy: {test_acc:.4f}")

    # =====================
    # ROC (CORRECT VERSION)
    # =====================
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
    plt.plot([0,1],[0,1],'k--')
    plt.legend()
    plt.title("ROC Curve")
    plt.show()

    # =====================
    # RETURN EVERYTHING
    # =====================
    return (
        all_labels,
        all_preds,
        all_probs,
        all_features,
        sample_images
    )

In [ ]:
import torch.fft

def get_frequency(x):
    # x: (B, C, H, W)
    x_gray = torch.mean(x, dim=1, keepdim=True)  # grayscale

    fft = torch.fft.fft2(x_gray)
    fft_shift = torch.fft.fftshift(fft)

    magnitude = torch.abs(fft_shift)
    magnitude = torch.log(magnitude + 1e-8)  # stabilize

    return magnitude

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class FSFDNet(nn.Module):
    def __init__(self):
        super(FSFDNet, self).__init__()

        # 🔹 Spatial Branch (EfficientNet-B0)
        self.spatial_model = models.efficientnet_b0(pretrained=True)
        self.spatial_model.classifier = nn.Identity()  # Output = 1280

        # 🔹 Frequency CNN (after FFT)
        self.freq_conv = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )  # Output = 64

        # 🔹 Edge CNN
        self.edge_conv = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )  # Output = 64

        # 🔹 Fusion Classifier
        self.fc = nn.Sequential(
            nn.Linear(1280 + 64 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,2)
        )

    def forward(self, x):

        # 🔸 Spatial features
        spatial_feat = self.spatial_model(x)

        # 🔸 Frequency branch (CORRECTED)
        x_gray = torch.mean(x, dim=1, keepdim=True)  # convert to grayscale
        fft = torch.fft.fft2(x_gray)
        fft_shift = torch.fft.fftshift(fft)
        fft_mag = torch.log(torch.abs(fft_shift) + 1e-8)

        freq_feat = self.freq_conv(fft_mag)
        freq_feat = freq_feat.view(freq_feat.size(0), -1)

        # 🔸 Edge branch
        grad_h = x_gray[:,:,1:,:] - x_gray[:,:,:-1,:]
        grad_w = x_gray[:,:,:,1:] - x_gray[:,:,:,:-1]

        grad_h = F.pad(grad_h, (0,0,0,1))
        grad_w = F.pad(grad_w, (0,1,0,0))

        edge = torch.abs(grad_h) + torch.abs(grad_w)

        edge_feat = self.edge_conv(edge)
        edge_feat = edge_feat.view(edge_feat.size(0), -1)

        # 🔸 Feature fusion
        fused = torch.cat([spatial_feat, freq_feat, edge_feat], dim=1)

        # 🔸 Classification
        out = self.fc(fused)
        return out


model = FSFDNet().to(device)

In [ ]:
def infer_image(model, image_path):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img)
        probs = F.softmax(output, dim=1)
        _, pred = torch.max(output,1)
    return "Real" if pred.item()==0 else "Fake", probs.cpu().numpy()[0]

In [ ]:
all_labels, all_preds, all_probs, all_features, sample_images = train_fsfd_full(
    model,
    train_loader,
    val_loader,
    test_loader
)

In [ ]:
# ============================================================
# FSFD-Net  —  PUBLICATION-GRADE VISUALIZATION SUITE  v5.2
#
# PATCHES vs v5.1:
#  1. Image titles now show full 4-line label:
#       ✓/✗ Original Label: <cls>
#       Real Prob: 0.9807
#       Fake Prob: 0.0193
#       Predicted Label: <cls>
#  2. _img_title() helper centralises this label — one place to edit
#  3. fig9 row height slightly increased to accommodate taller titles
# ============================================================

import os, random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, precision_score,
    recall_score, f1_score,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

os.makedirs("results/figures", exist_ok=True)

# ============================================================
# GLOBAL STYLE
# ============================================================
matplotlib.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          9,
    "axes.titlesize":     9,
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "axes.linewidth":     0.7,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.18,
    "legend.fontsize":    8.5,
    "legend.frameon":     True,
    "legend.framealpha":  0.92,
    "legend.edgecolor":   "#cccccc",
})

REAL_COLOR  = "#2196F3"
FAKE_COLOR  = "#F44336"
CORR_COLOR  = "#43A047"
WRONG_COLOR = "#FB8C00"

# ============================================================
# STEP 0 — MODEL / DEVICE GUARD
# ============================================================
_nb = globals()

if 'model' not in _nb:
    raise NameError(
        "\n\n  [FSFD-Net Viz] `model` is not defined.\n"
        "  Run in a prior cell:\n"
        "      model = FSFDNet(...).to(device)\n"
        "      model.eval()\n"
    )

if 'device' not in _nb:
    try:
        device = next(model.parameters()).device
        print(f"[INFO] device inferred from model: {device}")
    except StopIteration:
        device = torch.device('cpu')
else:
    device = _nb['device']

model = _nb['model']
model.eval()
print(f"[INFO] model  : {type(model).__name__}")
print(f"[INFO] device : {device}")

# ============================================================
# STEP 1 — COLLECT ALL OUTPUTS FROM THE TEST SET
# ============================================================

for _v in (
    'all_labels','all_preds','all_probs','all_features',
    'all_labels_list','all_preds_list','all_probs_list',
    'all_features_list','all_images_list',
    'sample_images','sample_labels','sample_preds',
    'sample_probs','sample_features',
):
    globals().pop(_v, None)
del _v

# ---- ✏️  Only line you ever need to edit ----
DATASET_ROOT = "/kaggle/working/subset_dataset_10k"
# ---------------------------------------------
BATCH_SIZE   = 32

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

test_dataset = datasets.ImageFolder(root=DATASET_ROOT, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"[INFO] Classes       : {test_dataset.classes}")
print(f"[INFO] class_to_idx  : {test_dataset.class_to_idx}")
print(f"[INFO] Total samples : {len(test_dataset)}")

_c2i           = test_dataset.class_to_idx
FAKE_CLASS_IDX = _c2i['Fake']
REAL_CLASS_IDX = _c2i['Real']

CLASS_NAMES = [''] * len(_c2i)
for name, idx in _c2i.items():
    CLASS_NAMES[idx] = name

print(f"[INFO] CLASS_NAMES       : {CLASS_NAMES}")
print(f"[INFO] FAKE_CLASS_IDX    : {FAKE_CLASS_IDX}")

all_images_list   = []
all_labels_list   = []
all_preds_list    = []
all_probs_list    = []
all_features_list = []

model.eval()
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs_dev = imgs.to(device)

        outputs = model(imgs_dev)
        if isinstance(outputs, tuple):
            logits, features = outputs[0], outputs[1]
        else:
            logits   = outputs
            features = None

        probs = F.softmax(logits, dim=1)[:, FAKE_CLASS_IDX].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()

        all_images_list.extend(imgs.cpu().numpy())
        all_labels_list.extend(lbls.numpy())
        all_preds_list.extend(preds)
        all_probs_list.extend(probs)
        if features is not None:
            all_features_list.extend(features.cpu().numpy())

print(f"[INFO] Collected {len(all_labels_list)} samples")
assert len(all_labels_list) > 0, "DataLoader returned no samples — check DATASET_ROOT."

all_labels   = np.array(all_labels_list)
all_preds    = np.array(all_preds_list)
all_probs    = np.array(all_probs_list)
all_features = np.array(all_features_list) if all_features_list else None

# ============================================================
# STEP 2 — RANDOM SAMPLING  (6 images, 3 real + 3 fake)
# ============================================================

def _draw_random_samples(images_list, labels_list, preds_list,
                          probs_list, features_list, n=6, seed=None):
    rng = random.Random(seed)
    np.random.seed(seed)

    indices  = list(range(len(labels_list)))
    real_idx = [i for i in indices if labels_list[i] == REAL_CLASS_IDX]
    fake_idx = [i for i in indices if labels_list[i] == FAKE_CLASS_IDX]

    half   = n // 2
    chosen = (rng.sample(real_idx, min(half, len(real_idx))) +
              rng.sample(fake_idx, min(half, len(fake_idx))))

    remaining = [i for i in indices if i not in set(chosen)]
    chosen   += rng.sample(remaining, max(0, n - len(chosen)))
    chosen    = chosen[:n]
    rng.shuffle(chosen)

    feats = ([features_list[i] for i in chosen]
             if features_list is not None and len(features_list) else None)
    return (
        [images_list[i] for i in chosen],
        [labels_list[i] for i in chosen],
        [preds_list[i]  for i in chosen],
        [probs_list[i]  for i in chosen],
        feats,
    )

(sample_images_raw,
 sample_labels_list,
 sample_preds_list,
 sample_probs_list,
 sample_features_raw) = _draw_random_samples(
    all_images_list, all_labels_list, all_preds_list,
    all_probs_list,
    all_features_list if all_features_list else None,
    n=6, seed=None,
)

sample_images   = np.array(sample_images_raw)
sample_labels   = np.array(sample_labels_list)
sample_preds    = np.array(sample_preds_list)
sample_probs    = np.array(sample_probs_list)
sample_features = (np.array(sample_features_raw)
                   if sample_features_raw is not None else None)

N = len(sample_images)   # 6
print(f"[INFO] Sampled {N} images for figures  "
      f"(Real={int((sample_labels==REAL_CLASS_IDX).sum())}, "
      f"Fake={int((sample_labels==FAKE_CLASS_IDX).sum())})")

# ============================================================
# HELPERS
# ============================================================

def _to_hwc_float(img):
    if isinstance(img, torch.Tensor):
        img = img.permute(1, 2, 0).cpu().numpy()
    else:
        img = np.transpose(img, (1, 2, 0))
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)


def _label_str(idx):
    return CLASS_NAMES[int(idx)]


def _fft_magnitude(gray_u8):
    f = np.fft.fftshift(np.fft.fft2(gray_u8.astype(np.float64)))
    return np.log1p(np.abs(f)).astype(np.float32)


def _dct_spectrum(gray_u8):
    h, w = gray_u8.shape
    out  = np.zeros((h, w), np.float32)
    b    = 8
    for r in range(0, h - b + 1, b):
        for c in range(0, w - b + 1, b):
            blk = gray_u8[r:r+b, c:c+b].astype(np.float32)
            out[r:r+b, c:c+b] = np.log1p(np.abs(cv2.dct(blk)))
    return out


def _residual(gray_u8):
    blur = cv2.GaussianBlur(gray_u8.astype(np.float32), (5, 5), 0)
    r    = gray_u8.astype(np.float32) - blur
    mn, mx = r.min(), r.max()
    return (r - mn) / (mx - mn + 1e-8)


def _sharpen_cam(gcam, low_pct=40):
    thresh = np.percentile(gcam, low_pct)
    gcam   = np.clip(gcam - thresh, 0, None)
    if gcam.max() > 1e-8:
        gcam /= gcam.max()
    gcam_u8 = np.uint8(255 * gcam)
    gcam_sm = cv2.bilateralFilter(gcam_u8, d=9, sigmaColor=75, sigmaSpace=75)
    return gcam_sm.astype(np.float32) / 255.0


def _border(ax, color, lw=2.0):
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_edgecolor(color)
        s.set_linewidth(lw)


def _place_row_labels(fig, axes_col0, labels, fontsize=8):
    fig.canvas.draw()
    for ax, text in zip(axes_col0, labels):
        pos = ax.get_position()
        fig.text(
            pos.x0 - 0.008,
            pos.y0 + pos.height / 2,
            text, ha='right', va='center',
            fontsize=fontsize, fontweight='bold', rotation=90,
        )


def _img_title(ts, ps, fake_prob):
    """
    Consistent 4-line image title used across all figures.

    Example output:
        ✓ Original Label: Real
        Real Prob: 0.9807
        Fake Prob: 0.0193
        Predicted Label: Real

    fake_prob  = P(Fake) from softmax — stored in sample_probs / all_probs.
    real_prob  = 1 - fake_prob  (binary classification).
    """
    real_prob = 1.0 - float(fake_prob)
    mark = "✓" if ts == ps else "✗"
    return (
        f"{mark} Original Label: {ts}\n"
        f"Real Prob: {real_prob:.4f}\n"
        f"Fake Prob: {float(fake_prob):.4f}\n"
        f"Predicted Label: {ps}"
    )


# ============================================================
# FIGURE 1 — CONFUSION MATRIX
# ============================================================

def plot_confusion_matrix():
    cm      = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    acc     = np.mean(all_preds == all_labels)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5),
                             gridspec_kw={"wspace": 0.48})

    for ax, data, fmt, cmap, cbar_lbl, title in [
        (axes[0], cm,      'd',   'Blues',   'Count',            '(a) Raw Counts'),
        (axes[1], cm_norm, '.2%', 'Oranges', 'Per-class Recall', '(b) Normalized'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, linecolor='white',
                    cbar_kws={"shrink": 0.78, "label": cbar_lbl},
                    annot_kws={"size": 13}, ax=ax)
        ax.set_title(title, fontweight='bold', pad=9)
        ax.set_xlabel("Predicted Label", labelpad=7)
        ax.set_ylabel("True Label",      labelpad=7)
        ax.tick_params(length=0)

    fig.suptitle(
        f"FSFD-Net  —  Confusion Matrix  |  Accuracy = {acc:.4f}",
        fontsize=11, fontweight='bold', y=1.03,
    )
    plt.savefig("results/figures/fig1_confusion_matrix.pdf")
    plt.show()
    print("[OK] Fig 1 : Confusion Matrix")

plot_confusion_matrix()


# ============================================================
# FIGURE 2 — ROC + PRECISION-RECALL
# ============================================================

def plot_roc_pr():
    fpr, tpr, _ = roc_curve(all_labels, all_probs, pos_label=FAKE_CLASS_IDX)
    roc_auc     = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(all_labels, all_probs,
                                          pos_label=FAKE_CLASS_IDX)
    ap           = average_precision_score(all_labels, all_probs,
                                           pos_label=FAKE_CLASS_IDX)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.3),
                             gridspec_kw={"wspace": 0.36})

    axes[0].plot(fpr, tpr, color=FAKE_COLOR, lw=2,
                 label=f"FSFD-Net (AUC = {roc_auc:.4f})")
    axes[0].plot([0,1],[0,1], 'k--', lw=1, label="Random")
    axes[0].fill_between(fpr, tpr, alpha=0.10, color=FAKE_COLOR)
    axes[0].set(xlim=[0,1], ylim=[0,1.02],
                xlabel="False Positive Rate", ylabel="True Positive Rate",
                title="(a) ROC Curve")
    axes[0].title.set_fontweight('bold')
    axes[0].legend(loc="lower right")
    axes[0].grid(True, alpha=0.25, ls='--')

    axes[1].plot(rec, prec, color=REAL_COLOR, lw=2,
                 label=f"FSFD-Net (AP = {ap:.4f})")
    axes[1].fill_between(rec, prec, alpha=0.10, color=REAL_COLOR)
    axes[1].set(xlim=[0,1], ylim=[0,1.02],
                xlabel="Recall", ylabel="Precision",
                title="(b) Precision-Recall Curve")
    axes[1].title.set_fontweight('bold')
    axes[1].legend(loc="lower left")
    axes[1].grid(True, alpha=0.25, ls='--')

    fig.suptitle("FSFD-Net  —  Detection Performance Curves",
                 fontsize=11, fontweight='bold', y=1.03)
    plt.savefig("results/figures/fig2_roc_pr.pdf")
    plt.show()
    print("[OK] Fig 2 : ROC + PR")

plot_roc_pr()


# ============================================================
# FIGURE 3 — MULTI-DOMAIN FREQUENCY ANALYSIS
#   4 rows × 6 cols  (RGB | FFT | DCT | Residual)
# ============================================================

def plot_spatial_frequency_analysis():
    IMG_H = 2.4
    DOM_H = 1.9
    FIG_W = N * 2.4 + 1.1
    FIG_H = IMG_H + 3*DOM_H + 0.85

    fig = plt.figure(figsize=(FIG_W, FIG_H))
    gs  = gridspec.GridSpec(
        4, N, figure=fig,
        height_ratios=[IMG_H, DOM_H, DOM_H, DOM_H],
        hspace=0.52, wspace=0.05,
        left=0.11, right=0.985,
        top=0.90,  bottom=0.04,
    )

    ROW_LABELS = ["RGB Image", "FFT Spectrum", "DCT (8×8)", "Residual Noise"]
    CMAPS      = [None, 'inferno', 'viridis', 'hot']
    axes_g     = [[fig.add_subplot(gs[r, c])
                   for c in range(N)] for r in range(4)]

    for i in range(N):
        img  = _to_hwc_float(sample_images[i])
        gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        ts   = _label_str(sample_labels[i])
        ps   = _label_str(sample_preds[i])
        prob = float(sample_probs[i])
        bg   = CORR_COLOR if ts == ps else WRONG_COLOR

        panels = [img, _fft_magnitude(gray), _dct_spectrum(gray), _residual(gray)]

        for r in range(4):
            ax = axes_g[r][i]
            ax.imshow(panels[r], cmap=CMAPS[r], aspect='auto')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            if r == 0:
                ax.set_title(
                    _img_title(ts, ps, prob),
                    fontsize=6.5, color="white", backgroundcolor=bg,
                    pad=2, linespacing=1.3,
                )
                _border(ax, bg, lw=2.0)

    _place_row_labels(fig, [axes_g[r][0] for r in range(4)], ROW_LABELS)

    leg = [mpatches.Patch(color=CORR_COLOR,  label="Correct"),
           mpatches.Patch(color=WRONG_COLOR, label="Wrong")]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=8.5, bbox_to_anchor=(0.55, -0.01))
    fig.suptitle(
        "FSFD-Net  —  Multi-Domain Analysis: RGB | FFT | DCT | Residual",
        fontsize=11, fontweight='bold',
    )
    plt.savefig("results/figures/fig3_multidomain_analysis.pdf")
    plt.show()
    print("[OK] Fig 3 : Multi-Domain Frequency Analysis")

plot_spatial_frequency_analysis()


# ============================================================
# FIGURE 4 — GRAD-CAM++ EXPLAINABILITY
# ============================================================

def plot_gradcam_analysis(model, device):
    target_layer = [model.spatial_model.features[-1]]
    cam_engine   = GradCAMPlusPlus(model=model, target_layers=target_layer)

    IMG_H  = 2.4
    DOM_H  = 1.9
    FIG_W  = N * 2.4 + 1.1
    FIG_H  = IMG_H + 3*DOM_H + 0.85

    fig = plt.figure(figsize=(FIG_W, FIG_H))
    gs  = gridspec.GridSpec(
        4, N + 1, figure=fig,
        height_ratios=[IMG_H, DOM_H, DOM_H, DOM_H],
        width_ratios=[1.0]*N + [0.06],
        hspace=0.52, wspace=0.05,
        left=0.11, right=0.985,
        top=0.90,  bottom=0.04,
    )

    ROW_LABELS = [
        "Input Image",
        "Grad-CAM++ Overlay",
        "Activation Heatmap",
        "Residual × CAM",
    ]

    axes_g   = [[fig.add_subplot(gs[r, c]) for c in range(N)] for r in range(4)]
    cbar_axs = [fig.add_subplot(gs[r, N]) for r in range(4)]
    for r, cax in enumerate(cbar_axs):
        cax.set_visible(r == 2)

    for i in range(N):
        img_t = torch.tensor(sample_images[i]).unsqueeze(0).float().to(device)
        img   = _to_hwc_float(sample_images[i])
        gray  = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        ts    = _label_str(sample_labels[i])
        ps    = _label_str(sample_preds[i])
        prob  = float(sample_probs[i])
        bg    = CORR_COLOR if ts == ps else WRONG_COLOR

        raw_cam = cam_engine(
            input_tensor=img_t,
            targets=[ClassifierOutputTarget(FAKE_CLASS_IDX)],
        )[0]
        gcam = _sharpen_cam(raw_cam, low_pct=40)

        overlay     = show_cam_on_image(img, gcam, use_rgb=True,
                                        image_weight=0.45).astype(np.uint8)
        gcam_u8     = np.uint8(255 * gcam)
        _, thresh_m = cv2.threshold(gcam_u8, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh_m,
                                       cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (255, 230, 0), 1)
        overlay = overlay.astype(np.float32) / 255.0

        heatmap = cv2.applyColorMap(gcam_u8, cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        res_cam = _residual(gray) * gcam

        panels = [img, overlay, heatmap, res_cam]
        cmaps  = [None, None, None, 'hot']

        for r in range(4):
            ax = axes_g[r][i]
            ax.imshow(panels[r], cmap=cmaps[r], aspect='auto')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
            if r == 0:
                ax.set_title(
                    _img_title(ts, ps, prob),
                    fontsize=6.5, color="white", backgroundcolor=bg,
                    pad=2, linespacing=1.3,
                )
                _border(ax, bg, lw=2.0)

    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_axs[2])
    cb.set_label("Activation Intensity", fontsize=7.5, labelpad=4)
    cb.ax.tick_params(labelsize=6.5)

    _place_row_labels(fig, [axes_g[r][0] for r in range(4)], ROW_LABELS)

    leg = [mpatches.Patch(color=CORR_COLOR,  label="Correct"),
           mpatches.Patch(color=WRONG_COLOR, label="Wrong")]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=8.5, bbox_to_anchor=(0.53, -0.01))
    fig.suptitle(
        "FSFD-Net  —  Grad-CAM++ Explainability  |  Forgery Localisation",
        fontsize=11, fontweight='bold',
    )
    plt.savefig("results/figures/fig4_gradcam_explainability.pdf")
    plt.show()
    print("[OK] Fig 4 : Grad-CAM++ Explainability")

plot_gradcam_analysis(model, device)


# ============================================================
# FIGURE 5 — UNIFIED FEATURE EXTRACTION STRIP
# ============================================================

def plot_unified_feature_extraction():
    DOMAINS      = ["RGB", "FFT", "DCT", "Residual"]
    CMAPS        = [None, 'inferno', 'viridis', 'hot']
    PANEL_RATIOS = [2.2, 1.0, 1.0, 1.0]

    COL_W  = 2.3
    FIG_W  = N * COL_W + 1.3
    FIG_H  = 9.0

    fig = plt.figure(figsize=(FIG_W, FIG_H))

    outer_gs = gridspec.GridSpec(
        1, N, figure=fig,
        hspace=0.0, wspace=0.10,
        left=0.12, right=0.985,
        top=0.91,  bottom=0.07,
    )

    first_col_axes = []

    for i in range(N):
        img  = _to_hwc_float(sample_images[i])
        gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        ts   = _label_str(sample_labels[i])
        ps   = _label_str(sample_preds[i])
        prob = float(sample_probs[i])
        bg   = CORR_COLOR if ts == ps else WRONG_COLOR

        panels = [img, _fft_magnitude(gray), _dct_spectrum(gray), _residual(gray)]

        inner_gs = gridspec.GridSpecFromSubplotSpec(
            4, 1,
            subplot_spec  = outer_gs[0, i],
            height_ratios = PANEL_RATIOS,
            hspace        = 0.0,
        )

        for r in range(4):
            ax = fig.add_subplot(inner_gs[r, 0])
            ax.imshow(panels[r], cmap=CMAPS[r], aspect='auto')
            ax.set_xticks([]); ax.set_yticks([])
            _border(ax, bg, lw=1.8)

            if r == 0:
                ax.set_title(
                    _img_title(ts, ps, prob),
                    fontsize=6.0, color="white",
                    backgroundcolor=bg, pad=2, linespacing=1.2,
                )

            if i == 0:
                first_col_axes.append(ax)

    fig.canvas.draw()
    for ax, domain in zip(first_col_axes, DOMAINS):
        pos = ax.get_position()
        fig.text(
            pos.x0 - 0.012,
            pos.y0 + pos.height / 2,
            domain,
            ha='right', va='center',
            fontsize=9, fontweight='bold', rotation=90,
        )

    fig.canvas.draw()
    for i in range(N):
        cell = outer_gs[0, i].get_position(fig)
        fig.text(
            cell.x0 + cell.width / 2,
            cell.y0 - 0.030,
            f"#{i+1}",
            ha='center', va='top',
            fontsize=8, color='#444444',
        )

    leg = [mpatches.Patch(color=CORR_COLOR,  label="Correct prediction"),
           mpatches.Patch(color=WRONG_COLOR, label="Wrong prediction")]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=8.5, bbox_to_anchor=(0.55, 0.01))

    fig.suptitle(
        "FSFD-Net  —  Unified Feature Extraction  "
        "(each column = one image: RGB | FFT | DCT | Residual)",
        fontsize=11, fontweight='bold', y=0.97,
    )
    plt.savefig("results/figures/fig5_unified_feature_extraction.pdf")
    plt.show()
    print("[OK] Fig 5 : Unified Feature Extraction Strip")

plot_unified_feature_extraction()


# ============================================================
# FIGURE 6 — CONFIDENCE & SCORE ANALYSIS
# ============================================================

def plot_confidence_analysis():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2),
                             gridspec_kw={"wspace": 0.38})

    real_p = all_probs[all_labels == REAL_CLASS_IDX]
    fake_p = all_probs[all_labels == FAKE_CLASS_IDX]

    axes[0].hist(real_p, bins=30, color=REAL_COLOR, alpha=0.72,
                 edgecolor='white', lw=0.4, label='Real')
    axes[0].hist(fake_p, bins=30, color=FAKE_COLOR, alpha=0.72,
                 edgecolor='white', lw=0.4, label='Fake')
    axes[0].axvline(0.5, color='black', ls='--', lw=1.1, label='Decision boundary')
    axes[0].set(xlabel="P(Fake)", ylabel="Count",
                title="(a) Score Distribution by Class")
    axes[0].title.set_fontweight('bold')
    axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

    ok = (all_preds == all_labels)
    axes[1].hist(all_probs[ok],  bins=30, color=CORR_COLOR,  alpha=0.72,
                 edgecolor='white', lw=0.4, label='Correct')
    axes[1].hist(all_probs[~ok], bins=30, color=WRONG_COLOR, alpha=0.72,
                 edgecolor='white', lw=0.4, label='Wrong')
    axes[1].axvline(0.5, color='black', ls='--', lw=1.1)
    axes[1].set(xlabel="P(Fake)", ylabel="Count",
                title="(b) Score Distribution by Correctness")
    axes[1].title.set_fontweight('bold')
    axes[1].legend(); axes[1].grid(True, alpha=0.25, ls='--')

    idx  = np.argsort(all_probs)
    cols = [FAKE_COLOR if l == FAKE_CLASS_IDX else REAL_COLOR
            for l in all_labels[idx]]
    axes[2].scatter(range(len(idx)), all_probs[idx], c=cols, s=5, alpha=0.65)
    axes[2].axhline(0.5, color='black', ls='--', lw=1.1)
    axes[2].set(xlabel="Sample index (sorted)", ylabel="P(Fake)",
                title="(c) Sorted Prediction Scores")
    axes[2].title.set_fontweight('bold')
    rp = mpatches.Patch(color=REAL_COLOR, label='Real')
    fp = mpatches.Patch(color=FAKE_COLOR, label='Fake')
    axes[2].legend(handles=[rp, fp])
    axes[2].grid(True, alpha=0.25, ls='--')

    fig.suptitle("FSFD-Net  —  Confidence & Score Analysis",
                 fontsize=11, fontweight='bold', y=1.03)
    plt.savefig("results/figures/fig6_confidence_analysis.pdf")
    plt.show()
    print("[OK] Fig 6 : Confidence Analysis")

plot_confidence_analysis()


# ============================================================
# FIGURE 7 — FEATURE SPACE  (PCA + t-SNE)
# ============================================================

def plot_feature_space():
    if all_features is None or len(all_features) < 10:
        print("[!] No feature vectors — skipping Fig 7.")
        return

    n_pca    = min(50, all_features.shape[1])
    feat_r   = PCA(n_components=n_pca, random_state=42).fit_transform(all_features)
    pca2     = PCA(n_components=2, random_state=42)
    emb_pca  = pca2.fit_transform(all_features)
    var_exp  = pca2.explained_variance_ratio_.sum() * 100

    perp     = min(30, max(5, len(all_features) // 4))
    emb_tsne = TSNE(n_components=2, perplexity=perp,
                    learning_rate='auto', init='pca',
                    random_state=42, n_iter=1000).fit_transform(feat_r)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                             gridspec_kw={"wspace": 0.35})

    for ax, emb, method in [(axes[0], emb_pca, "PCA"),
                             (axes[1], emb_tsne, "t-SNE")]:
        for cls, name, color in [
            (REAL_CLASS_IDX, "Real", REAL_COLOR),
            (FAKE_CLASS_IDX, "Fake", FAKE_COLOR),
        ]:
            m = all_labels == cls
            ax.scatter(emb[m,0], emb[m,1], c=color, label=name,
                       s=16, alpha=0.72, edgecolors='white', lw=0.3)
        wrong = (all_preds != all_labels)
        if wrong.sum():
            ax.scatter(emb[wrong,0], emb[wrong,1],
                       facecolors='none', edgecolors=WRONG_COLOR,
                       s=55, lw=1.2, label='Misclassified', zorder=5)
        ax.set_title(f"Feature Space ({method})", fontweight='bold', pad=8)
        ax.set_xlabel(f"{method} Dim 1", labelpad=5)
        ax.set_ylabel(f"{method} Dim 2", labelpad=5)
        ax.legend(markerscale=1.3)
        ax.grid(True, alpha=0.2, ls='--')

    axes[0].set_xlabel(
        f"PCA Dim 1  (variance explained = {var_exp:.1f}%)", labelpad=5)
    fig.suptitle("FSFD-Net  —  Feature Embedding (PCA | t-SNE)",
                 fontsize=11, fontweight='bold', y=1.03)
    plt.savefig("results/figures/fig7_feature_space.pdf")
    plt.show()
    print("[OK] Fig 7 : Feature Space")

plot_feature_space()


# ============================================================
# FIGURE 8 — MEAN FFT SPECTRUM PER CLASS
# ============================================================

def plot_mean_fft_per_class():
    real_ffts, fake_ffts = [], []
    for i in range(N):
        img  = _to_hwc_float(sample_images[i])
        gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        bucket = real_ffts if sample_labels[i] == REAL_CLASS_IDX else fake_ffts
        bucket.append(_fft_magnitude(gray))

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.3),
                             gridspec_kw={"wspace": 0.22})
    mr = np.mean(real_ffts, axis=0) if real_ffts else None
    mf = np.mean(fake_ffts, axis=0) if fake_ffts else None

    for ax, data, title, cmap, is_diff in [
        (axes[0], mr,   "(a) Mean FFT — Real",      'inferno', False),
        (axes[1], mf,   "(b) Mean FFT — Fake",      'inferno', False),
        (axes[2], None, "(c) Difference (Fake−Real)", 'RdBu_r',  True),
    ]:
        if is_diff and mr is not None and mf is not None:
            diff = mf - mr
            vmax = np.abs(diff).max()
            im   = ax.imshow(diff, cmap=cmap, vmin=-vmax, vmax=vmax)
            lbl  = "Δ Log Magnitude"
        elif data is not None:
            im  = ax.imshow(data, cmap=cmap)
            lbl = "Log Magnitude"
        else:
            ax.axis('off'); continue

        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=lbl)
        cb.ax.tick_params(labelsize=7)
        ax.set_title(title, fontweight='bold', pad=7)
        ax.axis('off')

    fig.suptitle(
        "FSFD-Net  —  Mean FFT Spectrum Comparison  (Real vs Fake)",
        fontsize=11, fontweight='bold', y=1.03,
    )
    plt.savefig("results/figures/fig8_mean_fft.pdf")
    plt.show()
    print("[OK] Fig 8 : Mean FFT per Class")

plot_mean_fft_per_class()


# ============================================================
# FIGURE 9 — PREDICTION GALLERY  (2 × 3 for N=6)
# ============================================================

def plot_prediction_gallery():
    rows, cols = 2, N // 2
    # Slightly taller rows to comfortably fit the 4-line title
    fig, axes  = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 3.5),
                              gridspec_kw={"hspace": 0.72, "wspace": 0.08})

    for i, ax in enumerate(axes.flat):
        img  = _to_hwc_float(sample_images[i])
        ts   = _label_str(sample_labels[i])
        ps   = _label_str(sample_preds[i])
        prob = float(sample_probs[i])
        bg   = CORR_COLOR if ts == ps else WRONG_COLOR

        ax.imshow(img, aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(
            _img_title(ts, ps, prob),
            fontsize=7.5, color="white",
            backgroundcolor=bg, pad=2, linespacing=1.25,
        )
        _border(ax, bg, lw=2.0)

    fig.suptitle("FSFD-Net  —  Prediction Gallery (6 Random Samples)",
                 fontsize=11, fontweight='bold', y=1.02)
    leg = [mpatches.Patch(color=CORR_COLOR,  label="Correct"),
           mpatches.Patch(color=WRONG_COLOR, label="Wrong")]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               fontsize=8.5, bbox_to_anchor=(0.5, -0.05))
    plt.savefig("results/figures/fig9_prediction_gallery.pdf")
    plt.show()
    print("[OK] Fig 9 : Prediction Gallery")

plot_prediction_gallery()


# ============================================================
# FIGURE 10 — PER-CLASS METRICS BAR CHART
# ============================================================

def plot_classification_metrics():
    pp  = precision_score(all_labels, all_preds, average=None)
    rr  = recall_score(all_labels,    all_preds, average=None)
    ff  = f1_score(all_labels,        all_preds, average=None)
    mp  = precision_score(all_labels, all_preds, average='macro')
    mr_ = recall_score(all_labels,    all_preds, average='macro')
    mf  = f1_score(all_labels,        all_preds, average='macro')
    acc = np.mean(all_preds == all_labels)

    x, w = np.arange(len(CLASS_NAMES)), 0.24
    fig, ax = plt.subplots(figsize=(8, 4.6))

    for vals, offset, color, label in [
        (pp,  -w,  '#5C6BC0', 'Precision'),
        (rr,   0,  '#26A69A', 'Recall'),
        (ff,   w,  '#EF5350', 'F1-Score'),
    ]:
        bars = ax.bar(x + offset, vals, w, label=label,
                      color=color, edgecolor='white', lw=0.4)
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.3f}',
                        xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=7.5)

    for val, color, lbl in [
        (mp,  '#5C6BC0', f'Macro Prec. {mp:.3f}'),
        (mr_, '#26A69A', f'Macro Rec.  {mr_:.3f}'),
        (mf,  '#EF5350', f'Macro F1    {mf:.3f}'),
    ]:
        ax.axhline(val, color=color, ls='--', lw=1.0, alpha=0.6, label=lbl)

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, fontsize=10)
    ax.set_ylim([0, 1.16])
    ax.set_ylabel("Score", labelpad=5)
    ax.set_title(
        f"FSFD-Net  —  Per-Class Metrics  |  Accuracy = {acc:.4f}",
        fontweight='bold', pad=8,
    )
    ax.legend(fontsize=8, ncol=2, loc='upper right')
    ax.grid(True, axis='y', alpha=0.25, ls='--')
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)

    plt.tight_layout()
    plt.savefig("results/figures/fig10_per_class_metrics.pdf")
    plt.show()

    print("\n" + "="*52)
    print("  FSFD-Net  —  Classification Report")
    print("="*52)
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
    print(f"  Accuracy : {acc:.4f}  |  Macro F1 : {mf:.4f}")
    print("="*52)
    print("[OK] Fig 10: Per-Class Metrics")

plot_classification_metrics()

# ============================================================
print("\n  All figures saved to results/figures/")
print("="*52)

In [ ]:
-------------------------------------------Cross-Dataset Evaluation---------------------------------------------

In [ ]:
import os

base_path = "/kaggle/input/datasets/sachchitkunichetty/rvf10k/rvf10k"
valid_path = os.path.join(base_path, "valid")

In [ ]:
print(os.listdir(valid_path))

In [ ]:
import shutil

clean_path = "/kaggle/working/clean_rvf10k"
os.makedirs(clean_path, exist_ok=True)

for cls in ["fake", "real"]:
    src = os.path.join(valid_path, cls)
    dst = os.path.join(clean_path, cls)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

valid_path = clean_path

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_dataset = datasets.ImageFolder(valid_path, transform=transform)
print("Classes:", test_dataset.classes)

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
import torch

model.load_state_dict(torch.load("/kaggle/working/best_fsfdnet.pth"))
model.eval()
model.to(device)

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
import numpy as np

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print("RVF10K Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1],
    target_names=["Fake", "Real"]
))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)